In [1]:

import feedparser
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from datetime import datetime



In [2]:
# Step 1: Define RSS feeds for AI/ML/DS content
FEEDS = [
    "https://arstechnica.com/ai/feed/",
    "https://www.reddit.com/r/MachineLearning/.rss",
    "https://www.theverge.com/rss/ai-artificial-intelligence/index.xml",
    "https://www.datanami.com/feed/",
    "https://becominghuman.ai/feed",
    "https://blog.paperspace.com/rss/",
    "https://distill.pub/rss.xml",
    "https://bair.berkeley.edu/blog/feed.xml",
    "https://aibusiness.com/rss.xml",
    "https://www.techrepublic.com/rssfeeds/topic/artificial-intelligence/",
    "https://www.oreilly.com/radar/feed/index.xml",
    "https://flowingdata.com/feed/",
    "https://humansofdata.atlan.com/feed/",
    "https://statsandr.com/index.xml",
    "https://petewarden.com/feed/",
    "https://ninazumel.com/feed.xml",
    "https://towardsdatascience.com/feed",
    "https://huggingface.co/blog/feed.xml",
    "https://www.kdnuggets.com/feed",
    "https://www.analyticsvidhya.com/feed/",
    "https://towardsdatascience.com/feed",
    "https://dev.to/feed/tag/machinelearning"
    
   
]

In [3]:
# Step 2: Extract article URLs from feeds
def get_article_links(feeds):
    article_urls = set()
    for feed_url in feeds:
        feed = feedparser.parse(feed_url)
        print(f"[Info] Feed: {feed_url} → {len(feed.entries)} entries")
        for entry in feed.entries:
            article_urls.add(entry.link)
    print(f"[Summary] Total unique article URLs: {len(article_urls)}")
    return list(article_urls)





In [4]:
# Step 3: Scrape article content
def extract_article_content(url, max_words=1000):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        res = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(res.content, "html.parser")

        title_tag = soup.find("h1")
        title = title_tag.get_text(strip=True) if title_tag else "Untitled"

        paragraphs = soup.find_all("p")
        full_text = " ".join(p.get_text(strip=True) for p in paragraphs)
        words = full_text.split()

        return {
            "title": title,
            "url": url,
            "word_count": len(words),
            "content": " ".join(words[:max_words])
        }
    except Exception as e:
        print(f"[Error] Failed to scrape {url}: {e}")
        return None




In [5]:
# Step 4: Scrape and collect articles
def scrape_articles(min_articles=1000):
    urls = get_article_links(FEEDS)
    print(f"[Info] Found {len(urls)} URLs")

    articles = []
    for i, url in enumerate(urls):
        article = extract_article_content(url)
        if article:
            articles.append(article)
            print(f"[{i+1}] Scraped: {article['title'][:50]}...")
        if len(articles) >= min_articles:
            break
        time.sleep(0.5)  # polite scraping

    return articles



In [6]:

# Step 5: Save to CSV
def save_to_csv(articles):
    df = pd.DataFrame(articles)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"combined{timestamp}.csv"
    df.to_csv(filename, index=False, encoding="utf-8")
    print(f"[Saved] {len(articles)} articles to {filename}")

# Run it
articles = scrape_articles(min_articles=1000)
if articles:
    save_to_csv(articles)
else:
    print("[Warning] No articles were scraped.")



[Info] Feed: https://arstechnica.com/ai/feed/ → 20 entries
[Info] Feed: https://www.reddit.com/r/MachineLearning/.rss → 27 entries
[Info] Feed: https://www.theverge.com/rss/ai-artificial-intelligence/index.xml → 10 entries
[Info] Feed: https://www.datanami.com/feed/ → 10 entries
[Info] Feed: https://becominghuman.ai/feed → 10 entries
[Info] Feed: https://blog.paperspace.com/rss/ → 15 entries
[Info] Feed: https://distill.pub/rss.xml → 52 entries
[Info] Feed: https://bair.berkeley.edu/blog/feed.xml → 10 entries
[Info] Feed: https://aibusiness.com/rss.xml → 50 entries
[Info] Feed: https://www.techrepublic.com/rssfeeds/topic/artificial-intelligence/ → 20 entries
[Info] Feed: https://www.oreilly.com/radar/feed/index.xml → 15 entries
[Info] Feed: https://flowingdata.com/feed/ → 10 entries
[Info] Feed: https://humansofdata.atlan.com/feed/ → 10 entries
[Info] Feed: https://statsandr.com/index.xml → 86 entries
[Info] Feed: https://petewarden.com/feed/ → 10 entries
[Info] Feed: https://ninazumel